In [36]:
import pandas as pd
from scipy.stats import chi2_contingency
from itertools import combinations
from statsmodels.stats.multitest import multipletests

In [37]:
# import after saving these in notebook 07

root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/DataFrames/" # MAC
#root = "Y:/_Tobias/LSM980/Mitotic_Stopwatch/DataFrames/" # PC

df_augmin = pd.read_csv(root + "Cohort_for_stats_Augmin.csv")
df_USP28 = pd.read_csv(root + "Cohort_for_stats_USP28.csv")

Chi-square test of independence

Tests whether cell category distribution depends on time bin.

Null hypothesis: distribution of 0/1/2 is the same in all bins.

Alternative: distributions differ.

Used widely for contingency tables.

In [38]:
def getstatistics(counts):
    # Because your data are counts in a contingency table, the most statistically appropriate and stable method for pairwise bin comparisons is actually:
    # Chi-square test on the 3×2 table (categories × bins)
    # This avoids the multinomial regression instability entirely.

    table = counts.pivot_table(
    index = "category",
    columns = "time_bin",
    values = "count",
    aggfunc = "sum",
    fill_value = 0
    )
    print("Global results")
    print(chi2_contingency(table))
    
    
    expanded = counts.loc[counts.index.repeat(counts["count"])].drop(columns = "count")


    
    def compare_bins(counts_df, bin1, bin2):

        subset = counts_df[counts_df["time_bin"].isin([bin1, bin2])]
    
        table = subset.pivot_table(
            index="category",
            columns="time_bin",
            values="count",
            aggfunc="sum",
            fill_value=0
        )

        chi2, p, dof, expected = chi2_contingency(table)

        return p

    bins = counts["time_bin"].unique()

    results = []

    for b1, b2 in combinations(bins, 2):
        p = compare_bins(counts, b1, b2)
        results.append((b1, b2, p))
    
    pairwise = pd.DataFrame(results, columns=["bin1","bin2","p"])
    
    
    
    pairwise["p_adj"] = multipletests(pairwise["p"], method="fdr_bh")[1] # this is the Multiple testing correction (Benjamini–Hochberg (FDR))

    print("Pairwise results")
    print(pairwise)    

In [39]:
getstatistics(df_augmin)

Global results
Chi2ContingencyResult(statistic=18.51670925052411, pvalue=0.017669277630128894, dof=8, expected_freq=array([[ 0.22900763,  5.72519084, 12.13740458,  9.16030534,  2.7480916 ],
       [ 0.24427481,  6.10687023, 12.94656489,  9.77099237,  2.93129771],
       [ 0.52671756, 13.16793893, 27.91603053, 21.06870229,  6.32061069]]))
Pairwise results
          bin1       bin2         p     p_adj
0  000-030 min    030-060  0.595717  0.595717
1  000-030 min    060-090  0.200553  0.368437
2  000-030 min    090-120  0.331593  0.368437
3  000-030 min  > 120 min  0.050835  0.127088
4      030-060    060-090  0.031362  0.104539
5      030-060    090-120  0.001760  0.010776
6      030-060  > 120 min  0.002155  0.010776
7      060-090    090-120  0.255291  0.368437
8      060-090  > 120 min  0.301775  0.368437
9      090-120  > 120 min  0.314342  0.368437


In [40]:
getstatistics(df_USP28)

Global results
Chi2ContingencyResult(statistic=8.271589643548767, pvalue=0.40740024076316006, dof=8, expected_freq=array([[ 0.11413043,  1.71195652,  2.9673913 ,  1.55978261,  0.64673913],
       [ 0.84782609, 12.7173913 , 22.04347826, 11.58695652,  4.80434783],
       [ 2.03804348, 30.57065217, 52.98913043, 27.85326087, 11.54891304]]))
Pairwise results
        bin1         bin2         p     p_adj
0    030-060      060-090  0.712861  0.792068
1    030-060      090-120  0.672845  0.792068
2    030-060    > 120 min  0.091996  0.459982
3    030-060  000-030 min  0.586646  0.792068
4    060-090      090-120  0.913999  0.913999
5    060-090    > 120 min  0.069565  0.459982
6    060-090  000-030 min  0.518990  0.792068
7    090-120    > 120 min  0.183347  0.611158
8    090-120  000-030 min  0.471791  0.792068
9  > 120 min  000-030 min  0.284644  0.711611
